### Testing generation and comparing results for basic LLM, standard RAG, and this NIR pipeline

This notebook evaluates the following pipelines to determine the most effective approach for generating answers:
1. Basic LLM: uses a general world overview description as context and generates a narrative element based on the query and this text
2. Standard RAG: creates a vector database with text fragments and generates a narrative element based on the query and retrieved context
3. This NIR pipeline (version with a pre-stage plan generation): retrieves context from a graph and then, first, generates an answer plan based on the query and context; second, generates the final answer using the plan and context (based on the query)
4. This NIR pipeline (version without pre-stages): retrieves context from a graph and then generates an answer based on the query and context

**Metrics used:**

RAG efficiency and world consistency:
1. Faithfulness (RAGAS) – measures how factually consistent the generated answer is with the provided context. It evaluates whether the answer is fully grounded in the retrieved information and does not introduce unsupported or hallucinated statements. It is computed by comparing each claim in the generated answer against the retrieved context and checking whether it can be directly inferred from it. Higher values indicate that the model strictly follows the given context without adding external or fabricated information.
2. Answer Relevancy (RAGAS) – measures how relevant the generated answer is to the input question. It evaluates whether the response directly addresses the query without drifting into unrelated information. It is typically computed by comparing semantic similarity between the question and the generated answer. Higher scores indicate that the answer is well-aligned with the user’s intent.
3. Context Precision (RAGAS) – measures how relevant the retrieved context passages are to the question. It evaluates the proportion of useful retrieved information compared to all retrieved context. It is computed by checking which retrieved chunks are actually relevant for answering the query. Higher values indicate that the retrieval step returns mostly useful and non-noisy information.
4. Context Recall (RAGAS) – measures how well the retrieved context covers all the information needed to answer the question. It evaluates whether all necessary supporting facts are present in the retrieved context. It is computed by comparing the required information for a correct answer with the retrieved passages. Higher values indicate that the retrieval system successfully captures most or all relevant knowledge.
5. Answer Correctness (RAGAS) – measures the overall correctness of the generated answer compared to a reference (ground truth) answer. It evaluates both factual accuracy and semantic similarity to the expected response. It is typically computed using a combination of exact matching, semantic similarity (via embeddings), and factual consistency checks. Higher values indicate that the answer is not only relevant but also factually correct.
6. BERTScore (Generated Text vs World Description) – measures semantic similarity between the generated text and the world description. It evaluates how well the generated content aligns with the predefined world context in terms of meaning rather than exact wording. The metric is computed using contextual embeddings by matching tokens between the generated text and the world description and calculating precision, recall, and F1 over these matches. Higher values indicate stronger consistency of the generated output with the established world setting.
7. BERTScore (Generated Text vs Ground Truth) – measures semantic similarity between the generated text and the reference (ground truth) answer. It evaluates how closely the model’s output matches the expected correct response in meaning. The score is computed using contextual embeddings in the same way as above, by aligning tokens between generated and reference texts. Higher values indicate that the generated answer is closer in meaning to the ground truth, even if the wording differs.
8. World Consistency (LLM-based evaluation) – evaluates whether the generated text is consistent with the given world description using LLM as a judge. The model is prompted to assess if the generated narrative element could exist within the defined world rules, lore, and constraints (information based on world description), and outputs a continuous score between 0 and 1. Higher scores indicate that the generated text is coherent with the world setting and does not violate its established rules or context.

Text and generated narrative element quality:
1. Distinct-2 – measures lexical diversity of the generated text by computing the ratio of unique bigrams (2-grams) to the total number of bigrams. It evaluates how varied the text is in terms of local word sequences. Higher values indicate more diverse and less repetitive language, while lower values suggest repetitive or templated phrasing.
2. Repetition-2 – measures the level of repetition in the generated text by calculating how often bigrams (2-grams) are repeated within the output. It captures redundancy and looping patterns in phrasing. Lower values indicate more fluent and non-redundant text, while higher values suggest repetitive or overly formulaic generation.
3. MAUVE – measures the distributional similarity between the generated text and reference human-written text distributions. It evaluates how close the model’s output is to human-like text in a probabilistic embedding space. The metric compares clusters of token embeddings from both distributions and quantifies their divergence. Higher MAUVE scores indicate that generated text is more similar to human-written text in style and structure.
4. Self-BLEU – measures diversity within a set of generated texts by treating each generated sample as a hypothesis and the rest as references. It computes BLEU scores across generated outputs to evaluate how similar they are to each other. Lower values indicate higher diversity among generated samples, while higher values suggest that outputs are too similar or repetitive across generations.
5. Interestingness (LLM-as-a-judge) – evaluates how interesting, engaging, and creatively meaningful the generated text is using an LLM as a judge. The metric outputs a score from 0 to 1. It considers several factors: (1) whether the content appropriately reflects choice and agency when the task allows it, (2) how diverse and stylistically appropriate the text is, including whether it fits the tone, style, and world of the game without being overly dramatic or inconsistent, and (3) how creative and novel the idea is, including whether it provides fresh insights, new information about the world, or unique player experiences. Higher scores indicate more engaging, original, and well-aligned narrative content.

In [ ]:
#imports

import os
import sys
import tqdm
import pandas as pd
import logging
import warnings

#some important stuff setup

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(project_root)
sys.path.insert(0, project_root)
BASE_OUTPUT = os.path.join(project_root, "assets/outputs/test_results")
os.makedirs(BASE_OUTPUT, exist_ok=True)

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("faiss").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")

#this nir imports

from nir.llm.manager import ModelManager
from nir.llm.providers import ModelConfig

from nir.tests.test_datasets import TEST_DATA_TEXT1, TEST_DATA_TEXT2, TEST_DATA_TEXT3
from nir.tests.evaluator import analyze_generation

from nir.graph.graph_storages.networkx_graph import NetworkXGraph
from nir.core.context_retriever import form_context_with_llm
from nir.core.answers_generator import generate_plan
from nir.core.answers_generator import filter_context
from nir.core.answers_generator import generate_answer_based_on_plan
from nir.core.answers_generator import generate_answer_based_on_context

In [ ]:
#models setup

manager = ModelManager()

instruct_model_config = ModelConfig(model_name="hf.co/VlSav/Vikhr-Nemo-12B-Instruct-R-21-09-24-Q4_K_M-GGUF:latest", temperature=0.0)
instruct_llm = manager.create_chat_model(name="evaluation_model_tests", option="ollama", config=instruct_model_config)

answer_model_config = ModelConfig(model_name="llama3.2:latest", temperature=0.7)
answer_llm = manager.create_chat_model(name="generation_model_tests", option="ollama", config=instruct_model_config)

embeddings_model = manager.create_embedding_model(name="embeddings", option="hf_local", model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
#data setup

test_data_lore_description = TEST_DATA_TEXT1
test_data_design_document = TEST_DATA_TEXT2
test_data_scenario = TEST_DATA_TEXT3

**Testing basic LLM**

In [ ]:
all_metrics_lore_description = []
generated_lore_description = []

for task in tqdm.tqdm(test_data_lore_description["tasks"], desc=f"Testing basic llm generation on lore description"):
    query = task["query"]
    reference = task["reference"]
    category = task.get("category", "default")

    context = test_data_lore_description.get("text_summary", "")

    prompt = f"Use provided context to answer user's query.\nContext:\n{context}\nQuery:\n{query}"
    answer_final = answer_llm.invoke(prompt)
    
    generated_lore_description.append({ "category": category, "generated_text": answer_final })
    
    metrics = analyze_generation(
        generated_text=answer_final,
        context=context,
        lore_summary=test_data_lore_description.get("text_summary", ""),
        reference_text=reference,
        query=query,
        category=category,
        llm=instruct_llm,
        embedding_model=embeddings_model,
        language="en"
    )

    all_metrics_lore_description.append(metrics)

In [ ]:
if not all_metrics_lore_description:
    display(pd.DataFrame({"status": ["No data for analysis"]}))
else:
    df = pd.DataFrame(all_metrics_lore_description)
    num_cols = df.select_dtypes(include="number").columns.tolist()
    if "category" in df.columns and len(df) > 0:
        cat_df = df.groupby("category")[num_cols].mean().reset_index()
    else:
        cat_df = df[num_cols].mean().to_frame().T
        cat_df["category"] = "default"
    overall = {col: df[col].mean() for col in num_cols}
    overall["category"] = "OVERALL"
    overall_df = pd.DataFrame([overall])

    final_df = pd.concat([cat_df, overall_df], ignore_index=True)
    cols_order = ["category"] + sorted([c for c in final_df.columns if c != "category"])
    final_df = final_df[cols_order]

    display(final_df.style.format(precision=4))

if generated_lore_description:
    gen_df = pd.DataFrame(generated_lore_description)
    styled_gen = gen_df.style.set_properties(
        subset=["generated_text"],
        **{
            "white-space": "normal",
            "overflow-wrap": "break-word", 
            "vertical-align": "top", 
            "text-align": "left",
            "max-width": "1200px", 
            "padding": "8px" 
        }
    ).hide(axis="index") 
    display(styled_gen)
else:
    display(pd.DataFrame({"status": ["No generated answers"]}))